In [54]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import numpy as np
import copy
import os
import configparser
import json
import time
import re

In [2]:
pd.set_option('display.max_colwidth', None)

* geo_loc: geograafilised kohad 
* object_loc: objektid, mis võivad olla kohad 
* org_loc: organisatsioonid, mis võivad olla kohad 
* event_loc: tegevused/sündmused, millel on korraga nii aja kui koha tähendus
* per_loc: inimene kui koht 
* abstract_loc: abstraktsed kohad, mille asukoht on kas ebamäärane või ei eksisteerigi

## Vastustega df

In [3]:
df = pd.read_csv("../gpt_output/n80_examples_large_v1_gpt_v1_10K_b12_v1.csv", encoding="utf-8", sep="|")

In [246]:
df.head(2)

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
0,12979894,autodesse,auto,müüma,NaN,ill,8089054,“ Nad tassivad raskeid mitmekiloseid pakke ja müüvad lehti otse keset liiklust autodesse .,NaN,NaN,NaN,yes,"The term 'autodesse' refers to cars, which are considered a physical location, so it is classified as yes."
1,18564119,põõsas,põõsas,kükitama,NaN,in,11594114,"Kõik need praegused tähtsad tegelased kükitasid põõsas ja ootasid aega , et istuda toolile , mille teised olid kätte võidelnud .",NaN,NaN,NaN,yes,NaN


In [5]:
saving_columns = ["head_id", "form", "lemma", "verb", "verb_compound", "morph_case", "sentence_id", "sentence", "timex_tag", "ekilex_tag", "ner_tag"]

In [162]:
len(list(df["lemma"].unique()))

5081

In [6]:
print(list(df["lemma"].unique()))

['auto', 'põõsas', 'vitriin', 'MTV', 'töökoht', 'peatus', 'Hiina', 'dzhungel', 'maneež', 'Kuressaare', 'That', 'ajakirjandus', 'haigla', 'Kaubanduskeskus', 'maja', 'USA', 'motopood', 'Kaasan', 'liit', 'Tartu', 'Kambodža', 'kaugõpe', 'jaamahoone', 'type9', 'Kenema', 'elutuba', 'play-offi', 'liikmesriik', 'transformaator', 'Pirita', 'pea', 'Nevada', 'erasfäär', 'Dubai', 'Panga', 'Invernessi', 'haldusõigus', 'sport', 'Kuuba', 'pori', 'püünis', 'Ameerika', 'koduküla', 'Korintos', 'kabiin', 'Riia', 'stuudio', 'korter', 'kool', 'Eesti', 'kraadiõpe', 'alevik', 'agent', 'hoidla', 'metsapiirkond', 'suund', 'tualett', 'kelder', 'väikeapteek', 'Slovakkia', 'igapäevaelu', 'Argentina', 'Räpina', 'rohukapp', 'boks', 'Jurmala', 'huul', 'köis', 'foto', 'Calgary', 'faas', 'maa', 'vallamaja', 'pirukas', 'kolle', 'hõbe', 'autoturg', 'Peking', 'laager', 'C-grupp', 'põhjaosa', 'London', 'rutiin', 'NLKP', 'kava', 'Nicaragua', 'Austraalia', 'kodu', 'firma', 'metallikool', 'Norra', 'kelleg', 'lennuk', 'lasteh

In [189]:
counts2 = df.groupby(["lemma"], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)
counts2

,lemma,count
506,Eesti,56
1399,Tallinn,52
2936,kodu,45
1064,Moskva,41
3051,kool,37
...,...,...
2016,ekraan,1
2017,eks,1
2019,eksamiruum,1
2020,ekskrement,1


In [234]:
counts2.iloc[200:210]

,lemma,count
2890,kiri,7
1498,Viimsi,7
3049,koobas,7
3270,küsimus,7
2212,fuajee,7
762,Kabul,7
2671,kallas,7
4506,stuudio,7
3445,lootus,7
4263,raudteejaam,7


In [ ]:
# algus -> ajamäärus? kordub 21 korda andmetes
# aeg -> ?? 13 korda

# mille alla kuuluks "lähedus"? "lõpp". tee lõpus, nädala lõpus

In [ ]:
# mis võiks olla tabelis isikud

In [113]:
with open("../../base_data/v05_wordlists_obl/alive.txt", encoding="utf-8") as f:
    alive = f.readlines()
    
alive = [e.strip() for e in potential_per]

In [114]:
pot_alive = []

for elem in list(df["lemma"].unique()):
    if elem in alive:
        pot_alive.append(elem)
pot_alive        

['mina',
 'müürsepp',
 'klient',
 'kunstnik',
 'Raul',
 'tema',
 'Ackermann',
 'kaitseliitlane',
 'ema',
 'õde',
 'vend',
 'arst',
 'kobakäpp']

### abstract_loc
* ebamäärased suunad/teekonnad: ida, trajektoor, liikus ummikteel 
* nähtamatud/abstraktsed/määratlemata piirideta alad: Wifi, kvantmaailm, arvutiturg, õhuruum, digitaalplatvorm, liigub läheduses, hommikukaste, rambivalgus
* veebisaidid, telekanalid: Delfi, Yle
* abstraktsed mõisted: kirjanduses liiguvad väited, lahkusin poliitikast/võimult, kasutusaladel käib testimine
* ülekantud tähendusega füüsiline liikumine: istus hooaja jooksul peatreeneripingile - sai peatreeneriks, istus lavastajapuldis - oli lavastaja


In [241]:
potential_abs = ["lääs", "popmuusika","ETV", "hämar", "valgus","ringkond","valdkond","teadvus", "trajektoor","teekond", "rööbas",  "ummiktee", "internet", "kvantmaailm", "arvutiturg", "õhuruum", "digitaalplatvorm", "läheduses", "hommikukaste", "rambivalgus", "Delfi", "kirjandus", "poliitikia", "võim", "kasutusala"]

undes = ["raamatupida", "elamu", "maja", "juht", "server", "firma", "valgusti", "ajakirjandusväljaanne"]
abs1 = df[(df["lemma"].str.contains('|'.join(potential_abs))) | (df["lemma"].isin(["ida", "veeb", "elu", "küsimus"])) | (df["form"].str.contains('|'.join(["mõtetes"])))]
abs1 = abs1[~(abs1["lemma"].str.contains('|'.join(undes)))]
abs1 = abs1.sample(frac=1)
abs1 = abs1.iloc[:100]
abs1 = abs1[saving_columns]
abs1

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag
4235,18398659,Läänes,lääs,ehitama,NaN,in,11485675,"Ja kas olete mõelnud , et kui oleksite selle automaadi ehitanud Läänes , oleksite juba ammu miljonär ?",NaN,location,LOC
428,20067631,ETV-s,ETV,askeldama,NaN,in,12537802,"“ See , et Toomas Lepp üldse nii kaua ETV-s peadirektorina askeldas , on väljakutse tervele mõistusele , ” kommenteeris ringhäälingunõukogu liige Andrus Herkel .",NaN,location,ORG
7719,5607005,valdkondadesse,valdkond,pidama,NaN,ill,3481993,"Haapsalu linnapea Teet Kallasvee ( Res Publica ) pidas pealinna kolleegi sekkumist kõigisse linna valdkondadesse ohtlikuks , kuid mitte üllatavaks .",NaN,NaN,NaN
1147,1304372,alateadvuses,alateadvus,säilima,NaN,in,817620,Ent alateadvuses on probleemid säilinud .,NaN,NaN,NaN
5328,28622426,ajakirjanduses,ajakirjandus,vastutama,NaN,in,18957487,Ametlikult vastutab ajakirjanduses avaldatu eest väljaandja .,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
3700,11945442,ajakirjandusest,ajakirjandus,tellima,NaN,el,7439888,"Hansatee tellib ajakirjandusest artikleid , jne.",NaN,NaN,NaN
2286,23217348,ETV-s,ETV,eksisteerima,NaN,in,14737914,"“ Eha Vettikku fenomen ” 6 . juulil ETV-s Et ka riigitelevisiooni fenomen veel eksisteerib , kinnitas lõppeval nädalal Eesti Telefilmi esilinastus .",NaN,location,ORG
7577,16502711,eluvaldkonnas,eluvaldkond,arenema,NaN,in,10282696,"Oli aeg , kus ühel kuuendikul maakerast arenes uusi suurepäraseid võrseid igas eluvaldkonnas .",NaN,NaN,NaN
3101,3818270,valdkonda,valdkond,laienema,NaN,adit,2382701,"Endiselt on ka sügisel ja talvel aktuaalne teksakangas , mis on arenenud ja laienenud igasse valdkonda .",NaN,NaN,NaN


In [242]:
len(abs1)

100

In [243]:
abs1["lemma"].unique()

array(['lääs', 'ETV', 'valdkond', 'alateadvus', 'ajakirjandus', 'Delfi',
       'küsimus', 'poolhämarus', 'kultuuriringkond', 'ringkond', 'elu',
       'kirjandus', 'ida', 'valgus', 'internet', 'ehitusvaldkond',
       'ajalookirjandus', 'internetiilm', 'hämarus', 'eluvaldkond',
       'teadvus', 'internetipood', 'mõte', 'internetiportaal',
       'õhtuhämarus', 'veeb', 'eneseteadvus', 'rööbas',
       'ajakirjanduspilt', 'IT-valdkond', 'popmuusika', 'hommikuvalgus',
       'lastekirjandus'], dtype=object)

In [244]:
abs1.to_csv("abstract_loc/abstract_loc_testset100.csv", sep="|", encoding="utf-8", index=False)

### geo_loc
* kohanimed: Bristol, Sepphoris
* ehitised/äride füüsilised asukohad: pangamaja, multimeediastuudio, Kuku klubi, käisime arvutifirmas
* alad, mille geograafiline asukoht on defineeritav: põlengupaik, põhjapoolus, kaldapealne, tagaots, tolmupilv
* koju

In [131]:
potential_loc = ["maja", "kabinet", "stuudio", "koht", "klubi", "firma", "paik", "pealne", "poolus", "Aafrika", "Eesti", "Narva", "Tartu"]

undes = ["protsess", "esikoht","aukoht", "majandus", "pidamine", "rida", "mida", "kohtumine", "aeg", "majapidamine", "liidri", "võistlus", "ehitus", "Riigi", "majand", "kohtuinstants", "omanikfirma", "konverents", "kaalumaja"]
geo1 = df[(df["lemma"].str.contains('|'.join(potential_loc)))]
geo1 = geo1[~(geo1["lemma"].str.contains('|'.join(undes)))]
geo1 = geo1.sample(frac=1)
geo1 = geo1.iloc[:100]
geo1 = geo1[saving_columns]
geo1

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag
1740,19063090,ööklubisse,ööklubi,algama,NaN,ill,11913401,"Ronitakse hõbedaselt läikivasse Hansabussi ja algab sõit Riiga , La Rocca ööklubisse , kus finaalkontsert peetakse .",NaN,location,NaN
6118,299501,Välis-Eestis,Välis-Eesti,säilima,NaN,in,180141,"Tänu neile on Välis-Eestis säilinud arvestatav hulk perekondi , kus keele puhtuse nõue on edasi antud vanemailt lastele .",NaN,location,LOC
8819,19829859,teemajas,teemaja,hargnema,NaN,in,12386887,Tegevus hargnes Siberi teemajas .,NaN,location,NaN
5105,7846578,klubis,klubi,õpetama,NaN,in,4887237,33aastane Ni õpetab ühes Luksemburgi klubis lapsi .,NaN,NaN,NaN
4192,10334047,Õnnetuspaika,õnnetuspaik,kiirustama,NaN,adit,6427967,Õnnetuspaika kiirustanud Soome ja Eesti päästekopterid meest enne pimeduse saabumist ei leidnud .,NaN,location,NaN
...,...,...,...,...,...,...,...,...,...,...,...
849,20710932,Lääne-Eestis,Lääne-Eesti,sadama,maha,in,12946583,Nädal hiljem aga sadas Lääne-Eestis maha paks lumi .,NaN,location,LOC
9648,17146871,elumajas,elumaja,pooma,üles,in,10700263,Tallinnas Sõpruse puiestee elumajas poos end üles 1972. aastal sündinud Juri .,NaN,location,NaN
143,8602930,Eestis,Eesti,jalutama,ringi,in,5363591,""" Seni arvasid piiritagused ratturid , et Eestis jalutavad ringi jääkarud .",NaN,location,LOC
9199,8379920,kohtadesse,koht,sokutama,NaN,ill,5223906,Tulevärgi korraldajad usaldasid iseenda pürotehnilisi oskusi ja sokutasid tuldpurskavaid asjandusi terrassil kõikvõimalikesse kohtadesse .,NaN,NaN,NaN


In [132]:
geo1["lemma"].unique()

array(['ööklubi', 'Välis-Eesti', 'teemaja', 'klubi', 'õnnetuspaik',
       'maakoht', 'peidupaik', 'Eesti', 'paik', 'maja', 'eramaja',
       'välisklubi', 'kalamaja', 'telemaja', 'töökoht', 'koht',
       'hukkumispaik', 'seltsimaja', 'asupaik', 'Tartu', 'vallamaja',
       'näitusemaja', 'Narva', 'kabinet', 'puumaja', 'autofirma',
       'kodukoht', 'hullumaja', 'dotcom-firma', 'Narva-Jõesuu',
       'viietärniöömaja', 'hulgimüügifirma', 'koolimaja', 'asulakoht',
       'firma', 'parkimiskoht', 'arhitektuuristuudio', 'partnerfirma',
       'peokoht', 'suusaklubi', 'kontserdipaik', 'majatiib', 'Eestimaa',
       'erakabinet', 'postimaja', 'Lääne-Eesti', 'elumaja', 'tapamaja'],
      dtype=object)

In [133]:
len(geo1["lemma"].unique())

48

In [134]:
geo1.to_csv("geo_loc/geo_loc_testset100.csv", sep="|", encoding="utf-8", index=False)

### per_loc

In [153]:
potential_per = ["kobakäpp","juut", "ema", "vanaema","vanaisa", "tädi", "onu", "õde","tema", "mina", "meie", "teie", "nemad", "kaitseliitlane", "isa", "õde", "vend", "arst", "maalane", "müürsepp", "klient", "kunstnik", "muusik", "Raul", "Ackermann", "naaber", "kaaslane"]

undes = ["kabinet","planeering","hoidlane","kleit",  "juhtme","sild", "olematu","süvend", "Iisaku", "teema", "saal", "mõle", "keel", "ala", "korter", "saar" , "kool", "tund", "klass", "ruum", "hoone", "muusika", "sarnane", "kinnis", "eelarve", "kodu", "paik", "maa", "kond", "Kenema", "nimekiri", "talu", "sadam", "riik"]
per1 = df[(df["lemma"].isin(potential_per)) | (df["lemma"].str.contains('|'.join(["maalane", "kaaslane", "kapten", "lane"])))]
per1 = per1[~(per1["lemma"].str.contains('|'.join(undes)))]
per1 = per1.sample(frac=1)
per1 = per1.iloc[:100]
per1 = per1[saving_columns]
per1

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag
5696,12238301,nendest,tema,sõitma,edasi,el,7629174,"Kindel on , et nendest edasi ega kõrvale su rong ei sõida .",NaN,NaN,NaN
9881,27274323,emast,ema,saama,välja,el,17890756,"Alates 22. nädalast kui see loode emast välja saab , siis seda protsessi nimetatakse juba sünnituseks .",NaN,NaN,NaN
7859,11509709,minust,mina,pääsema,välja,el,7155509,"Alles siis , kui ta oli oma õhinal tahtmises juba päris sihil , pääses minust lõpuks välja hele karjatus .",NaN,NaN,NaN
6944,11449055,vennas,vend,hoidma,kinni,in,7117988,"Sel ajal hoidis va vennas Imelill käsualust ohvrit jalgadest kinni , et õeksed saaksid tema laubahaava nagu kord ja kohus pikkade nõeltega kinni õmmelda .",NaN,NaN,NaN
2494,14061142,eestlasele,eestlane,jooksma,NaN,all,8759984,"Tõsine huvi Aigar Leoki vastu vallandus pärast tänavuste Euroopa noorte meistrivõistluste võitu , eestlasele jooksid tormi mitmed meeskonnad .",NaN,alive,NaN
8316,714846,juutidest,juut,jõudma,tagasi,el,456269,"NSV Liitu küüditatud ja evakueerunud juutidest jõudis pärast sõda tagasi ainult umbes 1000 Eesti juuti , umbes 2000 inimest ilmselt hukkus .",NaN,alive,NaN
960,5410857,prantslasest,prantslane,röövima,NaN,el,3363170,"DUBAI , 31. august ( Reuters-EPLO ) - Iraagis kaks prantslasest ajakirjanikku röövinud kurjategijad andsid Prantsusmaale veel 24 tundi aega tühistamaks moslemite pearättide kandmise keeld riigikoolides .",NaN,NaN,NaN
1372,10526312,klientidest,klient,tellima,NaN,el,6543390,"On siis põhjuseks kõrge hind või lihtsalt hirm uue tehnika ees , kuid numbrid räägivad selget keelt : kui algselt tellis öise nägemise seadme lisavarustusena ligi 20% klientidest , siis tänaseks on see protsent langenud kümme korda .",NaN,alive,NaN
7982,1310149,leedulasest,leedulane,röövima,NaN,el,821266,"61-aastaselt leedulasest bussijuhilt elu röövinud õnnetus toimus Eurolines Eesti juhi Hugo Osula sõnul eile kella kahe ajal pärastlõunal kohaliku aja järgi Salacgriva asula läheduses , 14-15 kilomeetri kaugusel Eesti piirist .",NaN,alive,NaN
6773,3505863,temas,tema,helisema,NaN,in,2193416,"Debora on ennekõike muusik , temas helisevad meloodiad ja tormlevad rahutud , ent nõtked rütmid , mängumaneer on elegantne ja tehnika nauditav , tema viiulikeeltelt kerkib publiku poole inimlikku soojust .",NaN,NaN,NaN


In [154]:
len(per1)

37

In [155]:
per1["lemma"].unique()

array(['tema', 'ema', 'mina', 'vend', 'eestlane', 'juut', 'prantslane',
       'klient', 'leedulane', 'kobakäpp', 'Raul', 'albaanlane', 'kapten',
       'õde', 'arst', 'müürsepp', 'kunstnik', 'kaitseliitlane',
       'Ackermann'], dtype=object)

In [156]:
per1.to_csv(f"per_loc/per_loc_testset{len(per1)}.csv", sep="|", encoding="utf-8", index=False)

### event_loc

* kleidiproov, värbamine, haldusmenetlus, prostitutsiooniprotsess, suusatreening jne

In [21]:
with open("../../base_data/v05_wordlists_obl/event.txt", encoding="utf-8") as f:
    potential_events = f.readlines()
    
potential_events = [e.strip() for e in potential_events]

In [40]:
searchfor = ["proov", "värbamine", "menetlus", "protsess", "treening", "koosolek", "pidu", "rünnak", "sõit", "pulm", "sõda", "etendus", "peied", "laat", "näitus", "matk", "õnnetus", "reis", "teenistus", "jaht", "seik", "kogunemine", "hakkamine", "võtmine"]
undes = ["ruum", "maja", "konteiner", "saal", "teine", "karp", "piletisaba", "keskus", "Georgia", "toimetulek", "REMOTEHOST", "püks", "plaat", "paik", "komitee", "viljant", "POMM", "asutus", "tabel", "reklaam", "söökla", "klass", "Kapellskär", "poliitika", "liiga", "folkloor", "play", "EMEX", "linn", "koht", "võistlustuli", "park", "EMU", "raamat", "Ljantor", "EMOR", "kett", "kõnts", "võistlusala", "joove", "kava" ]
ev1 = df[(df["lemma"].str.contains('|'.join(potential_events))) &  ~(df["lemma"].str.contains('|'.join(undes)))]

ev1 = ev1.sample(frac=1)
ev1 = ev1.iloc[:100]
ev1 = ev1[saving_columns]
ev1

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag
4893,13440430,sõtta,sõda,kihutama,NaN,adit,8387044,"Samamoodi nagu LBJ-d kihutasid Vietnami sõtta tippnõunikud , on ka Clinton oma meeskonna lükata-tõugata .",NaN,event,NaN
8559,15750891,eksponaatidest,eksponaat,tellima,NaN,el,9816538,Muuseumid tellivad hologramme oma väärtuslikest eksponaatidest : hologrammid asendavad näitustel hinnalisi esemeid .,NaN,NaN,NaN
1620,330070,stseenis,stseen,vestlema,NaN,in,202247,E. lonesco “ Kiilaspäise lauljanna ” 11. stseenis vestlevad kaks abielupaari .,NaN,event,NaN
7494,12623277,ajalootundides,ajalootund,õpetama,NaN,in,7880241,438 Aime Õngo õpetas ateistliku kasvatustöö tegemist keskkooli ajalootundides .,NaN,event,NaN
6595,13466528,esituses,esitus,eksisteerima,NaN,in,8402651,"Järgnevalt tehakse katse välja selgitada , esiteks , kas skansioon eksisteerib regilaulu traditsioonilises esituses objektiivsemal tasandil kui üksiku inimese tajuotsus ; ja kui see on nii , siis teiseks , kuidas seda kirjeldada füüsikaliste näitajate abil .",NaN,event,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1661,1070799,Lahesõjas,lahesõda,jooma,NaN,in,673406,"Araabiamaade enamik jõi Lahesõjas USA ja liitlaste poolele , kuid pole teada , mis mängu hakkab mängima Venemaa , kuidas käituvad Balkani riigid .",NaN,NaN,NaN
8110,16399868,Krossivõistluselt,krossivõistlus,sõitma,NaN,abl,10220152,"Krossivõistluselt sõitis Lehtlatele appi Lauri ühe visama vastase isa Ainars Karro , kes oma tutvusi kasutades leidis üles ühe Riia allilmas mõjuka mehe .",NaN,event,NaN
290,19488463,plahvatuses,plahvatus,lõhkema,NaN,in,12171364,"Maikuises plahvatuses Sõle tänava ühiselamus lõhkes lõhkekeha 20-aastase nooruki käes , kes sai surma .",NaN,event,NaN
9762,21786173,eksamile,eksam,ronima,NaN,all,13624356,"Näiteks räägitakse nii mõneski koolis osa õpilasi lihtsalt pehmeks , et nad ei roniks teatud eksamile – viivad kooli keskmise asjata alla .",NaN,event,NaN


In [39]:
ev1["lemma"].unique()

array(['kahevõitlus', 'laulatus', 'plahvatus', 'koonduslaager', 'MM',
       'eurosari', 'MK-sari', 'aktusekõne', 'EM-sari', 'lahing', 'trenn',
       'etendus', 'stseen', 'lahesõda', 'festival', 'plahvatusvalm',
       'esitus', 'finaalseeria', 'Moskva-visiit', 'ekspositsioon',
       'finaalmäng', 'identiteedimuutus', 'kahevõistlus', 'MM-valikmäng',
       'koondisetrenn', 'MM-finaalturniir', 'kiirrünnak',
       'EM-valiktsükkel', 'EM', 'MM-sari', 'olümpiamängud', 'istung',
       'USA-visiit', 'konverents', 'kümnevõistlus', 'eliitturniir',
       'võistlus', 'võitlus', 'kokkupõrge', 'apellatsioonkaebus',
       'filmifestival', 'sõda', 'MM-ralli', 'duell', 'mäestikulaager',
       'laristamispillerkaar', 'terrorirünnak', 'treeningrühm',
       'MM-koond', 'katastroof', 'avamine', 'olümpia', 'autovõidusõit',
       'asutamiskonverents', 'MK-etapp', 'eksam', 'kontsertosa',
       'infotund', 'ajalootund', 'EMA', 'avaring', 'MM-kalender',
       'maailmasõda', 'krossivõistlus', 'MM-st

In [42]:
ev1.to_csv("event_loc/event_loc_testset100.csv", sep="|", encoding="utf-8", index=False)

### org_loc
* organisatsioonid/kollektiivid: istun valitsuses, käin ülikoolis, hokitrennis, liigun võrgustikesse, lahkun töökohalt, vormelimaailm (koolid, trennid, lasteaiad)

In [88]:
searchfor = ["trenn", "laager", "valitsus", "töökoht", "kool", "liit", "nõukogu"]
undes = ["koolkond", "koolimaja", "koolikoridor", "koolihoone", "koolihoov", "koolisöökla", "keskkooliaste", "otsa-kool", "ülikoolilinn", "valitsuskvartal", "monoliitsus", "poliitik", "eliit"]
org1 = df[(df["lemma"].str.contains('|'.join(searchfor))) &  ~(df["lemma"].str.contains('|'.join(undes)))]
org1 = org1.iloc[:100]
org1 = org1.sample(frac=1)
org1 = org1[saving_columns]
org1

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag
3906,16081681,koolidesse,kool,lähetama,NaN,ill,10024651,"Siinkohal tasub Tartu Ülikooli rektoril kahe käe näppudel ära lugeda , kui mitu noort õpetajat ta läinud aastal oma kodumaa koolidesse lähetas .",NaN,NaN,NaN
3066,8792865,koonduslaagris,koonduslaager,laskma,maha,in,5484471,"Malloth lasi ühe juudi maha 1943. aastal Theresienstadti koonduslaagris , mis asus praeguse Tšehhi territooriumil .",NaN,location,NaN
1344,10083043,kooli,kool,kandma,NaN,adit,6276069,"Rekordimees Heiki Ojasild Kesklinna kooli IV klassist kannab iga päev kodunt kooli ja koolist koju 6,8 kilo kaaluvat ranitsat .",NaN,NaN,NaN
5673,27046437,nõukogusse,nõukogu,juhtuma,NaN,ill,17706403,"Olles olnud eelmise koosseisu ajal Põllumajanduse ja Maaelu Krediteerimise sihtasutuse nõukogu liige ja teades , et see ei allu ega ole allunud Riigikontrolli kontrollile , söandan ma väita , et juhul kui selle fondi nõukogusse juhtub ka niisuguseid krutskitega inimesi , siis on seal võrdlemisi suuri võimalusi pehmelt öeldes pahategudeks , mida kindlasti ei oleks sellisel juhul , kui tegemist oleks avalik-õigusliku juriidilise isikuga , mis seaduse järgi oleks pidanud toimima hakkama 1. juunist .",NaN,location,NaN
6506,3401265,usukoolides,usukool,baseeruma,NaN,in,2129872,"Saudi-Araabia salaluure Istakhbarat oli juba 1995. aasta lõpul otsustanud alustada raha andmist Talibanile , mis toona baseerus peamiselt Pakistani usukoolides .",NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1444,15099134,koolides,kool,hoidma,kinni,in,9403082,"Niisiis tuleks alustada inglise , saksa ja prantsuse keele õpetajate massilisest ettevalmistamisest , luues ühtlasi muidugi tingimused , mis neid koolides kinni hoiaks ega sunniks otsima tasuvamat ja meeldivamat tööd näiteks tõlgina .",NaN,NaN,NaN
4236,13615892,kloostrikoolis,kloostrikool,õpetama,NaN,in,8494803,"Ta naases Vologdamaale ja õpetas pea pool aastat Zaozerski erakla kloostrikoolis ning valmistus preestriks saama , ent sunniti sealt lahkuma kui lahkhelide tekitaja ja ateist , kes muuseas propageeris Darwini õpetust .",NaN,location,NaN
903,12407471,laagritest,laager,pagema,NaN,el,7739072,Juhuste läbi jõudis Sondasse ka Vene laagritest pagenud ja valedokumente kasutanud isa .,NaN,NaN,NaN
845,5809650,Tehnikaülikoolis,Tehnikaülikool,tudeerima,NaN,in,3607649,Sügisest tudeerib tippujuja Tallinna Tehnikaülikoolis majandust .,NaN,NaN,ORG


In [89]:
org1["lemma"].unique()

array(['kool', 'koonduslaager', 'nõukogu', 'usukool', 'Tehnikaülikool',
       'laager', 'ülikool', 'põgenikelaager', 'linnavalitsus',
       'lumelaager', 'julgeolekunõukogu', 'omavalitsus', 'trenn',
       'teatriliit', 'merelaager', 'kommertskool', 'lõunalaager',
       'mäestikulaager', 'liit', 'septembrilaager', 'üldhariduskool',
       'puhastuslaager', 'aianduskool', 'kutsekool', 'vangilaager',
       'pedagoogikaülikool', 'keskkool', 'töökoht', 'algkool',
       'asjadevalitsus', 'erikool', 'kontsentratsioonilaager',
       'maavalitsus', 'suvelaager', 'metallikool', 'külakool',
       'spordikool', 'distsiplinaarlaager', 'valitsus', 'koondisetrenn',
       'põhikool', 'munitsipaalkool', 'kõrgkool', 'alpilaager',
       'kloostrikool'], dtype=object)

In [90]:
org1.to_csv("org_loc/org_loc_testset100.csv", sep="|", encoding="utf-8", index=False)

### object_loc

* füüsilised objektid: esikohapoodium, Kuu, varundusseade, sadul, pilv, põuetasku
* kui väljendatakse abstraktset nähtust, aga objekti geograafiline asukoht on ikka määratav (nt hirm käib luust läbi - tegelikult pole hirm luus, aga luu on ise ikkagi kindla asukohaga)


In [103]:
searchfor = ["seade", "poodium", "mänguasi", "pirukas", "tasku", "pilv", "karp", "masin", "lennuk", "käru", "vitriin", "sadul", "putka", "auto", "katel", "kott", "ketas", "päevik", "kasukas", "lehv", "medal", "uks", "ratas"]
undes = ["turg", "bussitasku", "autoinspektsioon", "vang", "masingam", "pesula", "Luksemburg", "Aluksne", "medalikolmik", "tehas", "salong", "äri", "firma", "keskus", "pood", "baas", "parkla", "hotell", "automaat", "autorikaitse", "avarii", "register", "teenindus", "autotee", "võidusõit", "esikuuks"]
obj1 = df[(df["lemma"].str.contains('|'.join(searchfor))) &  ~(df["lemma"].str.contains('|'.join(undes))) | df["form"].str.contains("silmadest") | df["form"].str.contains("näkku")]
obj1 = obj1.iloc[:100]
obj1 = obj1.sample(frac=1)
obj1 = obj1[saving_columns]
obj1

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag
2,6712022,vitriinides,vitriin,ilutsema,NaN,in,4170566,"Kunstnike fantaasia võtab silme eest kirjuks - vitriinides ilutsevad rohelised , sinised , kollased , lillelised , liblikamustriga jm.",NaN,NaN,NaN
4503,12695817,karpi,karp,sulgema,NaN,adit,7924683,"Nii et pärast kerget kohvikueinet tulime oma nõndanimetatud hotelli tagasi , maksime — peremehe rõõmuks , sest külalisi oli tal nii varasel aastaajal alles õige harva , oma toa eest järgmise hommikuni ja sulgesime end täies anonüümsuses sesse tüütult ja lohutavalt roosalillelise tapeediga karpi .",NaN,NaN,NaN
5308,1964719,autos,auto,olema,alles,in,1235891,Ka Jevgeni mobiiltelefon oli autos alles .,NaN,NaN,NaN
1197,2509163,lennukitelt,lennuk,kukkuma,alla,abl,1577268,Jäätunud veekamakad kukuvad lennukitelt alla ja maanduvad inimeste eluamajadele ning aedadesse .,NaN,NaN,NaN
4630,13436270,autosse,auto,investeerima,NaN,ill,8384613,Uude autosse investeerib Opel ligikaudu 300 miljonit Saksa marka .,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2497,9245685,teokarbist,teokarp,voolama,välja,el,5763409,"Ka vetejumala jalgade juures olevast teokarbist voolab välja vesi , mis valgub mööda kaskaadi astmeid allapoole .",NaN,NaN,NaN
5088,4639953,mänguasjadesse,mänguasi,kaduma,NaN,ill,2893878,"Elusatesse mänguasjadesse kaob aja jooksul usk , kuid neid , kes veel keskeaski ei julge öösel pimedas üle toa käia , on meie seas hämmastavalt palju .",NaN,NaN,NaN
1982,22450863,mänguasjast,mänguasi,voolama,välja,el,14102418,"2 ) mänguasjas sisalduvad vedelikud ja gaasid ei saavutaks sellist temperatuuri ega rõhku , et nad voolaksid mänguasjast välja muul juhul , kui on vajalik mänguasja funktsioneerimiseks , ning võiksid tekitada põletuse või muu kehalise vigastuse ohtu .",NaN,NaN,NaN
3063,4426858,puidukoorimismasinasse,puidukoorimismasin,kukkuma,NaN,ill,2760280,Keilas asuva ettevõtte territooriumil hukkus eile hommikul puidukoorimismasinasse kukkunud noor mees .,NaN,NaN,NaN


In [104]:
obj1["lemma"].unique()

array(['vitriin', 'karp', 'auto', 'lennuk', 'lennukikandja', 'masin',
       'uks', 'CD-karp', 'tootmisseade', 'seade', 'autoaken', 'reisikott',
       'tagaratas', 'katel', 'kilekott', 'mootorlennuk', 'pirukas',
       'rahakott', 'kõvaketas', 'lehv', 'väikeveoauto', 'välisuks',
       'jopetasku', 'käekott', 'silm', 'rahatasku', 'sõiduauto',
       'autojuhiluba', 'pesumasin', 'autorool', 'metallkarp', 'sportauto',
       'putka', 'klassipäevik', 'jakitasku', 'kasukas', 'tasku', 'kott',
       'korjanduskarp', 'medal', 'ratas', 'elektriauto', 'miilitsaauto',
       'pilv', 'puituks', 'autokatus', 'nägu', 'lihakäru', 'muusikamasin',
       'laudakarp', 'keevaveekatel', 'lumepilv', 'küljetasku', 'juhiuks',
       'teokarp', 'mänguasi', 'puidukoorimismasin'], dtype=object)

In [105]:
obj1.to_csv("object_loc/object_loc_testset100.csv", sep="|", encoding="utf-8", index=False)